# Customer Liability Coverage Reconciliation

This notebook walks through the portfolio demonstration: load a synthetic customer ledger, aggregate liabilities with DuckDB, inspect one public BTC wallet balance, and render a reconciliation report. The customer ledger is synthetic and the public wallet addresses are demo data, not any real exchange's reserves.

In [1]:
from pathlib import Path
import sys
import pandas as pd

sys.path.append(str(Path('..').resolve()))
ledger_path = Path('../data/customer_ledger.csv')
ledger = pd.read_csv(ledger_path)
ledger.head()

  customer_id asset     balance       as_of
0    C-000002   ETH    0.107018  2026-05-05
1    C-000002  USDC  310.910000  2026-05-05
2    C-000003  USDT   35.780000  2026-05-05
3    C-000004   ETH    2.060398  2026-05-05
4    C-000004  USDT    9.800000  2026-05-05

The ledger is long-form: one row per customer-asset balance. A customer can hold multiple assets, which is why the synthetic 100-customer population creates 180 ledger rows.

In [2]:
from src.ledger import aggregate_by_asset, load_liabilities

liabilities = aggregate_by_asset(load_liabilities(ledger_path)).reset_index()
liabilities

  asset total_balance  customer_count top_1pct_share
0   BTC           1.0              40            0.4
1   ETH          50.0              50            0.4
2  USDC      121000.0              60            0.4
3  USDT        5000.0              30            0.4

The top 1% share is intentionally high to mimic a retail exchange liability book with a fat tail.

In [3]:
from src.chain import fetch_btc_balance

btc_balance = fetch_btc_balance('1A1zP1eP5QGefi2DMPTfTL5SLmv7DivfNa')
print(f'BTC genesis address balance: {btc_balance} BTC')

BTC genesis address balance: 57.19331671 BTC


The full CLI also calls Etherscan for ETH, USDC, and USDT. Those calls require `ETHERSCAN_API_KEY`; tests mock them so CI never depends on live APIs.

In [4]:
from decimal import Decimal

demo_recon = pd.DataFrame([
    {'asset': 'BTC', 'customer_liabilities': Decimal('1'), 'on_chain_reserves': Decimal('57.19417084'), 'coverage_ratio': '5719.417%', 'status': 'OK', 'usd_value_at_risk': Decimal('0')},
    {'asset': 'ETH', 'customer_liabilities': Decimal('50'), 'on_chain_reserves': Decimal('229.57969015'), 'coverage_ratio': '459.159%', 'status': 'OK', 'usd_value_at_risk': Decimal('0')},
    {'asset': 'USDC', 'customer_liabilities': Decimal('121000'), 'on_chain_reserves': Decimal('120133.63'), 'coverage_ratio': '99.284%', 'status': 'WATCH', 'usd_value_at_risk': Decimal('866.37')},
    {'asset': 'USDT', 'customer_liabilities': Decimal('5000'), 'on_chain_reserves': Decimal('273.58'), 'coverage_ratio': '5.472%', 'status': 'BREACH', 'usd_value_at_risk': Decimal('4726.42')},
])
demo_recon

  asset customer_liabilities on_chain_reserves coverage_ratio  status usd_value_at_risk
0   BTC                    1       57.19417084     5719.417%      OK                 0
1   ETH                   50      229.57969015      459.159%      OK                 0
2  USDC               121000        120133.63       99.284%   WATCH            866.37
3  USDT                 5000           273.58        5.472%  BREACH           4726.42

In [5]:
'HTML report renderer writes the same fields to output/recon_YYYY-MM-DD.html.'

HTML report renderer writes the same fields to output/recon_YYYY-MM-DD.html.

In production, this would add controlled price sourcing, alerting, review sign-off, stronger wallet ownership evidence, and asset-specific reserve policies. For this portfolio demonstration, the useful signal is the control shape: source data, aggregate, compare reserves, flag exceptions, and preserve daily evidence.